# Practice: measuring a classifier properly
*AI from scratch, chapter 12* · lesson: https://ihor.sh/ai/12-measuring-a-model.html

On a rare-event version of the breast-cancer data: why accuracy misleads, the confusion matrix, precision and
recall across thresholds, ROC and precision-recall curves, a threshold chosen by what mistakes cost, and calibration.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Compute model accuracy at threshold 0.5 (where predicted class is scores >= 0.5) using metrics.accuracy_score(yte, scores >= 0.5).', 'Compute confusion matrix at threshold 0.5 using metrics.confusion_matrix(yte, scores >= 0.5).', 'Use metrics.accuracy_score(yte, scores >= 0.5) with predicted binary labels scores >= 0.5.', 'Use metrics.confusion_matrix(yte, scores >= 0.5) to produce the [[tn, fp], [fn, tp]] array.'], 2: ['Generate boolean array flag = scores >= t.', 'Return metrics.precision_score(yte, flag, zero_division=0), metrics.recall_score(yte, flag, zero_division=0).'], 3: ['Compute ROC AUC score using metrics.roc_auc_score(yte, scores).', 'Compute Average Precision (PR AUC) using metrics.average_precision_score(yte, scores).', 'auc = metrics.roc_auc_score(yte, scores)', 'ap = metrics.average_precision_score(yte, scores)'], 4: ['For each threshold t in ts, calculate false negatives FN: ((scores < t) & (yte == 1)).sum()', 'Calculate false positives FP: ((scores >= t) & (yte == 0)).sum()', 'Compute cost = 20 * FN + 1 * FP across all thresholds.', 'Find the threshold with minimum cost using best_t = ts[int(np.argmin(cost))].', 'Count FN as ((scores < t) & (yte == 1)).sum() and FP as ((scores >= t) & (yte == 0)).sum().', 'Total cost list: [20 * ((scores < t) & (yte == 1)).sum() + ((scores >= t) & (yte == 0)).sum() for t in ts]', 'Select best_t = ts[int(np.argmin(cost))].'], 5: ['Obtain out-of-fold predicted probabilities for class 1 using cross_val_predict(..., cv=5, method="predict_proba")[:, 1].', 'Compute prob_true, prob_pred = calibration_curve(y, oof, n_bins=5).', 'Plot (prob_pred, prob_true) against the diagonal reference line [0, 1].', 'oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y, cv=5, method="predict_proba")[:, 1]', 'prob_true, prob_pred = calibration_curve(y, oof, n_bins=5)']}
_ANSWERS = {1: 'YWNjX21vZGVsID0gbWV0cmljcy5hY2N1cmFjeV9zY29yZSh5dGUsIHNjb3JlcyA+PSAwLjUpCmNtID0gbWV0cmljcy5jb25mdXNpb25fbWF0cml4KHl0ZSwgc2NvcmVzID49IDAuNSk=', 2: 'ZmxhZyA9IHNjb3JlcyA+PSB0CnJldHVybiBtZXRyaWNzLnByZWNpc2lvbl9zY29yZSh5dGUsIGZsYWcsIHplcm9fZGl2aXNpb249MCksIG1ldHJpY3MucmVjYWxsX3Njb3JlKHl0ZSwgZmxhZywgemVyb19kaXZpc2lvbj0wKQ==', 3: 'YXVjID0gbWV0cmljcy5yb2NfYXVjX3Njb3JlKHl0ZSwgc2NvcmVzKQphcCA9IG1ldHJpY3MuYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUoeXRlLCBzY29yZXMp', 4: 'Y29zdCA9IFsyMCAqICgoc2NvcmVzIDwgdCkgJiAoeXRlID09IDEpKS5zdW0oKSArICgoc2NvcmVzID49IHQpICYgKHl0ZSA9PSAwKSkuc3VtKCkgZm9yIHQgaW4gdHNdCmJlc3RfdCA9IHRzW2ludChucC5hcmdtaW4oY29zdCkpXQpwbHQucGxvdCh0cywgY29zdCk7IHBsdC54bGFiZWwoInRocmVzaG9sZCIpOyBwbHQueWxhYmVsKCJjb3N0Iik7IHBsdC5zaG93KCk=', 5: 'b29mID0gY3Jvc3NfdmFsX3ByZWRpY3QobWFrZV9waXBlbGluZShTdGFuZGFyZFNjYWxlcigpLCBMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MTAwMCkpLCBYLCB5LCBjdj01LCBtZXRob2Q9InByZWRpY3RfcHJvYmEiKVs6LCAxXQpwcm9iX3RydWUsIHByb2JfcHJlZCA9IGNhbGlicmF0aW9uX2N1cnZlKHksIG9vZiwgbl9iaW5zPTUpCnBsdC5wbG90KFswLCAxXSwgWzAsIDFdLCAiLS0iLCBjb2xvcj0iZ3JheSIpOyBwbHQucGxvdChwcm9iX3ByZWQsIHByb2JfdHJ1ZSwgIm8tIik7IHBsdC54bGFiZWwoInByZWRpY3RlZCIpOyBwbHQueWxhYmVsKCJhY3R1YWwgc2hhcmUiKTsgcGx0LnNob3coKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

## The data: make the disease rare
In the original data 37% of tumors are malignant. Screening data is far more lopsided, so keep only 1 in 6 of the
malignant cases: about 9% positives. Label 1 = malignant (the thing we're looking for).

In [ ]:
d = load_breast_cancer()
y_all = 1 - d.target                                   # 1 = malignant
rng = np.random.default_rng(0)
keep = (y_all == 0) | (rng.random(len(y_all)) < 1 / 6)
X, y = d.data[keep], y_all[keep]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
scores = model.predict_proba(Xte)[:, 1]
print(len(y), "tumors,", y.mean().round(3), "malignant")

## 1. The accuracy trap
When positive cases are rare (here ~9%), a naive model that always predicts 0 (benign) achieves high accuracy:
`acc_lazy = metrics.accuracy_score(yte, np.zeros_like(yte))` (~91% accuracy, but completely useless!).

Now evaluate our actual logistic regression model at decision threshold 0.5:
- Set `acc_model`: the model's accuracy at threshold 0.5 using `accuracy_score`.
- Set `cm`: the 2×2 confusion matrix at threshold 0.5 using `confusion_matrix` (layout: `[[tn, fp], [fn, tp]]`).

In [ ]:
# A naive baseline predicting all benign (class 0) for every test patient:
acc_lazy = metrics.accuracy_score(yte, np.zeros_like(yte))

acc_model = cm = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print("Model accuracy:", acc_model, "Lazy baseline:", acc_lazy)
print("Confusion matrix:\n", cm)

In [ ]:
check(1, lambda: acc_lazy > 0.88, 
      f"Saying 'benign' to everyone is {acc_lazy:.1%} accurate, yet catches 0 real cancers.")
check(1, lambda: acc_model is not None and acc_model > 0.90, 
      f"Model accuracy ({acc_model}) should be > 0.90 at threshold 0.5.")
check(1, lambda: cm is not None and cm.sum() == len(yte) and cm[1].sum() == yte.sum(), 
      "Row 1 (second row) of the confusion matrix must sum to total actual malignant cases in yte.")

## 2. Precision and recall across thresholds
Write `pr_at(t)` returning `(precision, recall)` when flagging tumors with `scores >= t`.
Use `precision_score` and `recall_score` from `sklearn.metrics` with `zero_division=0`.
Lower thresholds catch more cases (higher recall) but raise more false alarms (lower precision).

In [ ]:
def pr_at(t):
    """Compute precision and recall at probability threshold t.
    
    Args:
        t: float threshold in [0, 1]. Tumors with scores >= t are predicted malignant (1).
    Returns:
        tuple (precision, recall): both floats in [0, 1].
    """
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

for t in (0.1, 0.3, 0.5, 0.7, 0.9):
    print("threshold", t, " precision, recall:", pr_at(t))

In [ ]:
check(2, lambda: pr_at(0.1)[1] >= pr_at(0.9)[1] and pr_at(0.9)[0] >= pr_at(0.1)[0], 
      "Lower threshold (0.1) must achieve higher recall, while higher threshold (0.9) achieves higher precision.")

## 3. ROC and precision-recall curves
Calculate ranking performance across all possible thresholds:
- Set `auc`: area under the ROC curve using `roc_auc_score(yte, scores)`.
- Set `ap`: average precision (area under the PR curve) using `average_precision_score(yte, scores)`.

The helper plots below show both curves. The PR curve's baseline is the dataset base rate (~9%).

In [ ]:
auc = ap = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
metrics.RocCurveDisplay.from_predictions(yte, scores, ax=ax[0])
metrics.PrecisionRecallDisplay.from_predictions(yte, scores, ax=ax[1])
plt.show()
print("AUC:", auc, "AP:", ap, "base rate:", yte.mean())

In [ ]:
check(3, lambda: auc is not None and auc > 0.95 and ap is not None and ap > yte.mean() * 5, 
      f"AUC ({auc}) should be > 0.95 and Average Precision ({ap}) should be far above base rate ({yte.mean():.3f}).")

## 4. Choose the threshold by cost
In clinical screening, errors have unequal costs:
- Missing a malignant tumor (a **False Negative**, FN: `scores < t` and `yte == 1`) costs **20** units (delayed treatment).
- An unnecessary biopsy (a **False Positive**, FP: `scores >= t` and `yte == 0`) costs **1** unit.

For every threshold in `ts`, compute the total cost: `20 * FN + 1 * FP`.
Set `best_t` to the cheapest threshold (where cost is minimized).

In [ ]:
ts = np.linspace(0.01, 0.99, 99)
best_t = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("cheapest threshold:", best_t)

In [ ]:
check(4, lambda: best_t is not None and best_t < 0.5, 
      "when missing a cancer costs 20× more than a biopsy, the best threshold moves well below 0.5")

## 5. Is 0.8 really 80%?
Calibration checks whether predicted probabilities correspond to real frequencies: when the model predicts 0.8,
are about 80% of those cases actually malignant?

Use `cross_val_predict` on `(X, y)` with 5-fold cross-validation and `method="predict_proba"` to obtain out-of-fold probabilities.
Then compute the curve with `calibration_curve(y, oof, n_bins=5)`.
Set `prob_true` and `prob_pred`.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_predict

prob_true = prob_pred = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print("predicted:", prob_pred, "actual share:", prob_true)

In [ ]:
check(5, lambda: prob_true is not None and len(prob_true) >= 3 and np.all(np.abs(prob_true - prob_pred) < 0.35), 
      "Logistic regression is reasonably calibrated: points (prob_pred, prob_true) lie close to the diagonal.")

## Next
Back to the path: https://ihor.sh/ai/ · next: probability 4, confidence and tests.